In [2]:
# ────────────────────────────────────────────────────────────────
# Cell 1: Quick environment check (updated for A6000 48GB)
# ────────────────────────────────────────────────────────────────

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CUDA in PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else "N/A")
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else "N/A")

# Clear cache for fresh start
torch.cuda.empty_cache()

PyTorch: 2.10.0+cu130
CUDA available: True
Device count: 1
Current device: 0
Device name: NVIDIA RTX A6000
CUDA in PyTorch: 13.0
cuDNN: 91200
VRAM (GB): 51.526500352


In [3]:
# ────────────────────────────────────────────────────────────────
# Cell 2: Imports
# ────────────────────────────────────────────────────────────────

import numpy as np
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [4]:
# ────────────────────────────────────────────────────────────────
# Cell 3: Load cleaned CSVs
# ────────────────────────────────────────────────────────────────

# Load the cleaned dataset 
dataset = load_dataset(
    "csv",
    data_files={
        "train": "./train.csv",
        "validation": "./val.csv",
        "test": "./test.csv",
    }
)

print(f"Train examples: {len(dataset['train']):,}")
print(f"Val examples:   {len(dataset['validation']):,}")
print(f"Test examples:  {len(dataset['test']):,}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train examples: 1,756,584
Val examples:   585,512
Test examples:  585,526


In [5]:
# ────────────────────────────────────────────────────────────────
# Cell: Convert "real" → 0 and "fake" → 1 
# ────────────────────────────────────────────────────────────────

def map_to_int(example):
    # Get label, clean it, make lowercase
    lbl = str(example["label"]).strip().lower()
    
    # Direct mapping — only two possibilities
    if "real" in lbl:
        example["label"] = 0
    elif "fake" in lbl:
        example["label"] = 1
    else:
        # Safety net: show problem immediately
        print(f"Warning: Unexpected label '{lbl}' → skipping conversion")
        # You can raise error instead if you want strictness:
        # raise ValueError(f"Label must be 'real' or 'fake', got '{lbl}'")
    
    return example


print("Converting 'real' → 0 and 'fake' → 1 ...")

dataset = dataset.map(
    map_to_int,
    num_proc=4,
    desc="Mapping labels to 0/1"
)

# Quick check — MUST show <class 'int'> now
print("\nAfter mapping (should be integers):")
for split in ["train", "validation", "test"]:
    if split in dataset and len(dataset[split]) > 0:
        first_label = dataset[split][0]["label"]
        print(f"{split:12} → label: {first_label} (type: {type(first_label).__name__})")

Converting 'real' → 0 and 'fake' → 1 ...


Mapping labels to 0/1 (num_proc=4):   0%|          | 0/1756584 [00:00<?, ? examples/s]

Mapping labels to 0/1 (num_proc=4):   0%|          | 0/585512 [00:00<?, ? examples/s]

Mapping labels to 0/1 (num_proc=4):   0%|          | 0/585526 [00:00<?, ? examples/s]


After mapping (should be integers):
train        → label: 1 (type: str)
validation   → label: 0 (type: str)
test         → label: 1 (type: str)


In [7]:
# Convert using pandas (very fast and clear)
from datasets import Dataset, disable_caching

disable_caching()
for split in dataset:
    df = dataset[split].to_pandas()
    df["label"] = df["label"].astype(int)           # <--- this line does it
    dataset[split] = Dataset.from_pandas(df)
    print(f"{split:12} → label type now: {type(dataset[split][0]['label'])}")

train        → label type now: <class 'int'>
validation   → label type now: <class 'int'>
test         → label type now: <class 'int'>


In [ ]:
from huggingface_hub import login
login("your_huggingface_token_here")  # Replace with your actual token

In [9]:
# ────────────────────────────────────────────────────────────────
# Cell 4: Tokenizer setup (same, but check lengths on new data)
# ────────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Optional: Print a few examples to understand real lengths (very useful!)
print("Checking token lengths of first 5 train examples...")
for i in range(5):
    tokens = tokenizer(dataset["train"][i]["text"], add_special_tokens=True)
    print(f"Example {i}: {len(tokens['input_ids'])} tokens")


MAX_LENGTH = 256  

def preprocess(examples, tokenizer):   
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        add_special_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=False,
    )
    tokenized["labels"] = examples["label"]
    return tokenized

# Right before your .map() call
dataset.cleanup_cache_files()
print("Cleaned up old cache files for this dataset.")

# ── changed part ──
tokenized_datasets = dataset.map(
    preprocess,
    batched=True,
    batch_size=1000,
    # num_proc=6,  # can stay >1
    desc="Tokenizing",
    remove_columns=dataset["train"].column_names,
    fn_kwargs={"tokenizer": tokenizer},     # ← this is the key line
)

# Dynamic padding → perfect choice
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,                  # or 'longest' — same effect here
    max_length=MAX_LENGTH          # helps trainer know bound
)

print("Tokenization done. Sample tokenized item keys:", tokenized_datasets["train"].features.keys())

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

c:\gyanendra\non_aug\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dr-PrashantGupta\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Checking token lengths of first 5 train examples...
Example 0: 441 tokens
Example 1: 79 tokens
Example 2: 77 tokens
Example 3: 140 tokens
Example 4: 184 tokens
Cleaned up old cache files for this dataset.


Tokenizing:   0%|          | 0/1756584 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/585512 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/585526 [00:00<?, ? examples/s]

Tokenization done. Sample tokenized item keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


# do not close name-gyanendra 

In [10]:
# ────────────────────────────────────────────────────────────────
# Cell 5: Model setup (same, num_labels=2)
# ────────────────────────────────────────────────────────────────
MODEL_NAME = "roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type="single_label_classification",
    torch_dtype=torch.bfloat16  # Use bf16 for A6000 efficiency
)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# ────────────────────────────────────────────────────────────────
# Cell 6: Metrics function (same)
# ────────────────────────────────────────────────────────────────

from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score, roc_auc_score, average_precision_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()  # prob of fake class
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'balanced_acc': balanced_accuracy_score(labels, preds),
        'mcc': matthews_corrcoef(labels, preds),
        'roc_auc': roc_auc_score(labels, probs),
        'pr_auc': average_precision_score(labels, probs)
    }

In [13]:
# ────────────────────────────────────────────────────────────────
# Cell 7: TrainingArguments (optimized for A6000 48GB)
# ────────────────────────────────────────────────────────────────

from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./roberta-fake-news-cleaned",
    
    # ─── Increased epochs ───
    num_train_epochs=7,                  
    
    # Batch & memory settings (already good for A6000)
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=8,      
    dataloader_num_workers=4,
    
    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    
    # Mixed precision
    bf16=True,
    
    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",           
    greater_is_better=True,
    save_total_limit=3,                   # keep last 3 checkpoints
                         
    
    # Efficiency
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

# ─── More robust early stopping ───
early_stop_callback = EarlyStoppingCallback(
    early_stopping_patience=3,            
    early_stopping_threshold=0.0001       # ← stop only if no improvement > 0.0008 in f1
)

callbacks = [early_stop_callback]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
# ────────────────────────────────────────────────────────────────
# Cell 8: Trainer setup (same, but with optimized args)
# ────────────────────────────────────────────────────────────────

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks
)

In [15]:
# Double-check keys (should only have input_ids, attention_mask, labels)
print("Sample item keys after tokenization:", tokenized_datasets["train"][0].keys())

Sample item keys after tokenization: dict_keys(['input_ids', 'attention_mask', 'labels'])


# do not close just Minize this window

name - gyanendra 

In [16]:
# ────────────────────────────────────────────────────────────────
# Cell 9: Train (same call, but optimized params will speed it up)
# ────────────────────────────────────────────────────────────────

print("\n" + "═"*70)
print("Starting local fine-tuning of RoBERTa-base on cleaned data")
print("Optimized for A6000 48GB: bf16, batch=64 (eff=512), grad_acc=8")
print("═"*70 + "\n")
  
trainer.train()


══════════════════════════════════════════════════════════════════════
Starting local fine-tuning of RoBERTa-base on cleaned data
Optimized for A6000 48GB: bf16, batch=64 (eff=512), grad_acc=8
══════════════════════════════════════════════════════════════════════



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Balanced Acc,Mcc,Roc Auc,Pr Auc
1,0.348108,0.029415,0.990198,0.988959,0.982859,0.985899,0.988493,0.978399,0.999274,0.998818
2,0.226675,0.025174,0.991780,0.994193,0.982159,0.988139,0.989544,0.981893,0.999516,0.999237
3,0.194410,0.021441,0.992984,0.991943,0.987900,0.989917,0.991803,0.984542,0.999599,0.999347
4,0.181865,0.020679,0.993173,0.992199,0.988189,0.990190,0.992015,0.984960,0.999627,0.999390
5,0.181238,0.020779,0.993194,0.992762,0.987679,0.990214,0.991913,0.985004,0.999627,0.999394
6,0.183305,0.020724,0.993184,0.992878,0.987533,0.990198,0.991871,0.984982,0.999631,0.999399
7,0.177561,0.020731,0.993182,0.992883,0.987523,0.990196,0.991867,0.984978,0.999630,0.999398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=12012, training_loss=0.3807724450494383, metrics={'train_runtime': 41661.994, 'train_samples_per_second': 295.139, 'train_steps_per_second': 0.288, 'total_flos': 1.6176183452397158e+18, 'train_loss': 0.3807724450494383, 'epoch': 7.0})

# Do not  close just Minize this window

In [17]:
# ────────────────────────────────────────────────────────────────
# Cell 10: Test evaluation (same)
# ────────────────────────────────────────────────────────────────

print("\nFinal test evaluation...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\nTest results:")
for k, v in test_results.items():
    if k.startswith("eval_"):
        print(f"{k.replace('eval_', '').ljust(12)} : {v:.4f}")


Final test evaluation...



Test results:
loss         : 0.0202
accuracy     : 0.9933
precision    : 0.9928
recall       : 0.9880
f1           : 0.9904
balanced_acc : 0.9921
mcc          : 0.9853
roc_auc      : 0.9997
pr_auc       : 0.9995
runtime      : 617.6955
samples_per_second : 947.9200
steps_per_second : 7.4070


In [18]:
# ────────────────────────────────────────────────────────────────
# Cell 11: Save model (updated path)
# ────────────────────────────────────────────────────────────────

save_path = "./roberta-fake-news-cleaned-best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved → {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → ./roberta-fake-news-cleaned-best
